# <font color="#418FDE" size="10" uppercase>**B: SimCLR, An Unsupervised Contrastive Model**</font>
----

> Last update: 20240311

By the end of this lecture, you will be able to:
* Explain SimCLR model of [Chen et al. (2020)](http://proceedings.mlr.press/v119/chen20j/chen20j.pdf).
* Develop SimCLR custom loss and training functions in TensorFlow.

## **1. SimCLR Pretext Model**

### **1.1. SimCLR in General**

> [Chen et al. (2020)](http://proceedings.mlr.press/v119/chen20j/chen20j.pdf) introduced SimCLR, a framework for unsupervised contrastive learning of visual representations, achieving significant accuracy improvements on benchmarks like CIFAR10, CIFAR100, and Caltech-101 through fine-tuning.

> The model consists of a deep encoder (e.g., a convolutional neural network) and a projector (a simpler neural network with dense layers) that transforms the encoder's output into a space where contrastive loss is applied.

> SimCLR operates by enhancing the similarity of representations from different augmentations of the same image while decreasing similarity between representations of different images. This process relies on a batch of augmented data, comparing every possible pair within a batch to learn useful features. The learned encoder is then adaptable to various downstream tasks.

> Notably, SimCLR's approach is versatile and extendable beyond visual data to other types, such as temporal sequences.


### **1.2. SimCLR Schematics & Mathematics**

> <div align="left">
  <img src="https://raw.githubusercontent.com/mhrgroup/course_self_supervised_learning/main/images/simclr1.png" width="50%">
  <br>
  <figcaption>Figure: SimCLR Pretext Schematic.</figcaption>
</div>

> The figure above presents the SimCLR pretext model schematically. Let's overview this schematic:
* **Input Images**: The process starts with a batch of input images. Each image ${X}$ is augmented twice to produce ${\tilde{X}_i}$ and ${\tilde{X}_j}$—a positive pair.
* **Encoder $f(\cdot)$**: Each augmented image is passed through an encoder—typically a convolutional neural network (CNN)—to extract features ${H_i}$ and ${H_j}$.
* **Projector $g(\cdot)$**: The encoder's output features are then projected into a lower-dimensional space using a neural network, resulting in vectors ${Z_i}$ and ${Z_j}$.
* **Contrastive Loss**: The objective is to maximize the agreement between ${Z_i}$ and ${Z_j}$ for the positive pair and minimize it for negative pairs. The contrastive loss is defined as:
\begin{equation}
\begin{aligned}
\mathcal{L}_{\text {contrastive }}=-\log \frac{\exp \left(\operatorname{sim}\left(Z_i, Z_j\right) / \tau\right)}{\sum_{k=1}^N 1_{[k \neq i]} \exp \left(\operatorname{sim}\left(Z_i, Z_k\right) / \tau\right)}
\end{aligned}
\end{equation}
where $\text{sim}(Z_i, Z_j)$ is the cosine similarity between vectors ${Z_i}$ and ${Z_j}$ and $\tau$ is a [temperature scaling parameter](https://www.youtube.com/watch?v=YjVuJjmgclU).
* **Downstream Task**: Post-training, the encoder can be adapted for downstream tasks like classification by adding a new layer, such as a SoftMax layer, on top of the encoder output.

> This architecture allows for the learning of representations by using the data's structure, without the need for labeled examples.

> The superior performance of SimCLR can be attributed to:
* The generation of two distinct augmentations for each input, which leads to more robust feature representations being captured by the encoder (specifically in the last hidden layer as shown in the figure).
* The employment of a straightforward projector that significantly transforms the encoder's output for the pretext task.

> During training, SimCLR's loss function works by pulling closer the representations of augmentations from the same data point and simultaneously pushing away the representations of augmentations from different data points within a batch.

### **1.3. SimCLR Algorithm**

> <div align="left">
  <img src="https://raw.githubusercontent.com/mhrgroup/course_self_supervised_learning/main/images/simclralgorithm0.png" width="30%">
  <br>
  <figcaption>Figure: SimCLR Algorithm.</figcaption>
</div>

> The figure above outlines the main learning algorithm of SimCLR, which is as follows:
* **Inputs**: The algorithm takes a batch size $N$, a temperature parameter $\tau$, and the structures for the encoder $f$, projector $g$, and augmentation set $\mathcal{T}$.
* **Augmentation**: For each data point ${x_k}$ in the sampled minibatch, two augmentation functions are sampled from the set $\mathcal{T}$ ($t$ & $t{'}$). Each data point is augmented twice to create two views: $\tilde{x}_{2k-1}$ and $\tilde{x}_{2k}$.
* **Feature Representation**: Each augmented data point is then passed through the encoder network $f$ to get the feature representations $h_{2k-1}$ and $h_{2k}$.
* **Projection**: These representations are then passed through the projector g to get the vectors $z_{2k-1}$ and $z_{2k}$, which will be used in the contrastive loss calculation.
* **Similarity Calculation**: Once all data points have been processed, the algorithm computes the pairwise similarity ${s_{i,j}}$ for all possible pairs using cosine similarity.
* **Contrastive Loss**: The contrastive loss $\ell(i, j)$ is calculated for each positive pair of projections using the softmax function, scaled by the temperature $\tau$. The loss is calculated such that it encourages positive pairs to be similar and negative pairs to be dissimilar.
* **Optimization**: The networks $f$ and $g$ are updated with the aim of minimizing the contrastive loss $\mathcal{L}$.
* **Output**: After training, the algorithm retains the encoder network $f(\cdot)$ and discards the projector $g(\cdot)$.

> The key idea is that the encoder learns to produce similar representations for augmented versions of the same image and dissimilar representations for different images, which helps in learning robust features from the data without requiring labels.

## **2. SimCLR Loss & Training Functions**


> In TensorFlow, custom loss functions necessitate a pair of inputs: the actual and predicted values. Since SimCLR is unsupervised, we fabricate placeholder actual outputs that match the dimensions of the predictions. These predictions are the projections from the projector, labeled as $z_{estimate}$. Considering a batch size of $N$, the $z_{real}$ matrix will have dimensions of ${2}\times{N}$. To expedite training through parallel processing and efficient memory use, the implementation of loss functions should avoid iterative loops. Custom loss functions ought to be expressed using TensorFlow's tensor operations to ensure compatibility and performance.

In [ ]:
#@title Custom SimCLR Loss Function
'''
Abbreviations:
    datain: Input data
    ind   : Index
    tf    : TensorFlow
'''

import tensorflow as tf

# The @tf.function decorator in TensorFlow converts a Python function into a
# TensorFlow graph-based function. This transformation allows the function to be
# executed more efficiently, leveraging TensorFlow's graph execution capabilities,
# which can lead to significant speedups.
@tf.function
def fun_simclr_loss(z_real, z_estimate):
    # The z_real parameter is not used since SimCLR is an unsupervised model.
    # This dummy variable is discarded.
    del z_real

    # Temperature parameter is set to 0.1. This hyperparameter can be tuned
    # for specific problems to control the smoothness of the output distribution.
    toe = .1

    # Calculate the number of projections, which is twice the batch size (2N).
    num = z_estimate.shape[0]

    # Generate two sets of indices for all possible pair combinations within the batch.
    # ind0 is a temporary variable holding the repeated range for generating indices.
    ind0 = tf.repeat(tf.expand_dims(tf.range(0, num), axis=0), num, axis=0)
    # Flatten ind0 to create a single list of indices for the first element of pairs.
    ind1 = tf.reshape(ind0, (num**2, 1))[:, 0]
    # Flatten the transpose of ind0 to create a single list of indices for the second element of pairs.
    ind2 = tf.reshape(tf.transpose(ind0), (num**2, 1))[:, 0]

    # The temporary index tensor is no longer needed after use.
    del ind0

    # Select the projections based on the first set of indices to form the first elements of pairs.
    vector_1 = tf.gather(z_estimate, ind1, axis=0)
    # The first set of indices is no longer needed after use.
    del ind1

    # Select the projections based on the second set of indices to form the second elements of pairs.
    vector_2 = tf.gather(z_estimate, ind2, axis=0)
    # The second set of indices is no longer needed after use.
    del ind2

    # Calculate the cosine similarity between each pair of projections and negate it to prepare for loss calculation.
    s = -tf.reshape(tf.keras.losses.cosine_similarity(vector_1, vector_2, axis=1), (num, num))

    # The vector tensors are no longer needed after computing similarities.
    del vector_1
    del vector_2

    # Calculate the nominator of the contrastive loss function for each pair.
    nom = tf.exp(s / toe)

    # Calculate the denominator of the contrastive loss function by excluding the self-similarity term.
    x1 = tf.exp(s / toe)
    x2 = 1 - tf.eye(num, dtype=tf.float32)
    denom = tf.repeat(tf.expand_dims(tf.math.reduce_sum(x1 * x2, axis=1), axis=1), num, axis=1)

    # The intermediate tensors used for the denominator are no longer needed.
    del s
    del x1
    del x2

    # Calculate the loss `l(i, j)` for each pair of projections using the computed nominator and denominator.
    l = -tf.math.log(nom / denom)

    # Cleanup intermediate tensors to save memory.
    del nom
    del denom

    # Prepare indices to extract the diagonal elements from the loss matrix, which correspond to the actual pair losses.
    ind_2k0 = tf.range(0, num, 2, dtype=tf.int32)  # Even indices
    ind_2k1 = tf.range(1, num, 2, dtype=tf.int32)  # Odd indices

    # Extract and sum the losses for the actual pairs using the prepared indices.
    loss_mat1_1 = tf.gather(l, ind_2k0, axis=0)
    loss_mat1_2 = tf.gather(loss_mat1_1, ind_2k1, axis=1)
    loss_mat1 = tf.linalg.diag_part(loss_mat1_2)

    loss_mat2_1 = tf.gather(l, ind_2k1, axis=0)
    loss_mat2_2 = tf.gather(loss_mat2_1, ind_2k0, axis=1)
    loss_mat2 = tf.linalg.diag_part(loss_mat2_2)

    # After extracting the necessary elements, the large loss tensor can be discarded.
    del l

    # Combine the individual loss components into a single tensor.
    loss_mat = loss_mat1 + loss_mat2

    # Compute the final loss by taking the sum over all individual losses and normalizing by the number of pairs.
    L = tf.math.reduce_sum(loss_mat) / num

    # Return the final computed loss.
    return L


In [ ]:
#@title SimCLR training function
'''
Abbreviations:
    datain: Input data
    ind   : Index
    tf    : TensorFlow
'''
import tqdm 

# Function definition for training a model using the SimCLR approach
def fun_train_simclr(model, dataset, fun_augment_01, fun_augment_02,
                     epochs=100, verbose=1, patience=3, learning_rate=0.001):

    # Determine the output size of the model's last layer
    z_size = model.layers[-1].weights[-1].shape[0]

    # Initialize a list to keep track of the loss values for each epoch
    loss = []

    # Initialize the optimizer with the specified learning rate
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    # Loop through each epoch
    for epoch in range(epochs):
        # Initialize a variable to keep track of the running loss for the current epoch
        loss_running = 0

        # Initialize a progress bar using tqdm to visualize training progress
        pbar = tqdm(dataset,
                    desc  = f"SimCLR Training: {epoch + 1:03d}/{epochs}", # Description shown in the progress bar
                    ncols = 125, # Width of the progress bar
                    leave = True) # Whether the progress bar should remain after completion

        # Iterate over each batch in the dataset
        for batch_num, batch in enumerate(pbar, 1):
            if isinstance(batch, tuple):
                batch_in = batch[0] # Get the input data from the batch
            else:
                batch_in = batch

            # Apply two different augmentations to the input data
            x_tilda_01 = fun_augment_01(batch_in)
            x_tilda_02 = fun_augment_02(batch_in)

            # Concatenate the augmented data along the batch axis and reshape it for the model input
            x_tilda = tf.reshape(tf.concat([x_tilda_01, x_tilda_02], axis=0),
                                (x_tilda_01.shape[0] * 2, # Corrected to ensure the correct shape for concatenation
                                 x_tilda_01.shape[1], x_tilda_01.shape[2],
                                 x_tilda_01.shape[3]))

            # Create a dummy output tensor of the same size as the model's output
            z_real = tf.random.uniform((x_tilda.shape[0], z_size))

            # Open a GradientTape to record the operations for automatic differentiation
            with tf.GradientTape() as tape:
                # Pass the concatenated and augmented inputs through the model
                z_estimate = model(x_tilda, training=True)
                # Calculate the batch loss using the custom SimCLR loss function
                loss_batch = fun_simclr_loss(z_real, z_estimate)

            # Calculate the gradients of the loss with respect to the model's trainable variables
            gradients = tape.gradient(loss_batch, model.trainable_variables)

            # Apply the calculated gradients to the model's variables to minimize the loss
            optimizer.apply_gradients(zip(gradients, model.trainable_variables))

            # Update the running loss for the epoch
            loss_running += loss_batch.numpy()  # Ensure loss is a scalar by calling .numpy()

            # If verbose, update the progress bar with the current average loss
            if verbose:
                pbar.set_postfix(loss = f"{loss_running/batch_num:.6f}")

        # Append the average loss for this epoch to the loss history
        loss.append(loss_running / batch_num)

        # Check for early stopping: if there's no improvement in loss for a specified number of epochs, stop training
        if epoch >= patience:
            if loss[-1] > min(loss[:-patience]):
                print("Early stopping due to no improvement in loss.")
                break  # Exit the training loop

    # Return the trained model and the history of loss values
    return model, loss


# <font color="#418FDE" size="10" uppercase>**B: SimCLR, An Unsupervised Contrastive Model**</font>
----

In this lecture, you learned to:
* Explain SimCLR model of [Chen et al. (2020)](http://proceedings.mlr.press/v119/chen20j/chen20j.pdf).
* Develop SimCLR custom loss and training functions in TensorFlow.

In the following lecture (lecture C), we will go over a SimCLR experiment.